In [1]:
# preprocess import numpy as np
from scipy.interpolate import griddata
from datetime import timedelta, datetime, date
import dateutil.parser
import pandas as pd
from scipy.spatial import distance_matrix
import xarray as xr
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


In [2]:
## 
country = 'croatia' ## FILL WITH THE COUNTRY NAME e.g. cuba, croatia
data_path   = '' ## FILL WITH THE PATH of OBSERVATIONAL data
coords_path = '../data/' ## FILL WITH THE PATH OF THE FILE WITH THE FINAL COORDINATES
data_path   = f'{data_path}allstat_solar_radiation.csv'
coords_path = f'{coords_path}country_fin_coordinates.nc'

output_path = f'../data/idw_data/{country}/' ## FILL WITH THE PATH of desidered OUTPUT PATH
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)

### Loading the IDW's final coordinates:

In [3]:
#---------------------------------------------------------------------------------------
# High resolution grid obtained as a downscaling output
# Latitudes & Longitudes are stored in a .nc files with the ones of the other countries.
#---------------------------------------------------------------------------------------

def files_reading(country_name):
    file_coords     = xr.open_dataset(coords_path)
    lon_fin = (file_coords[f'longitude_{country_name}'].values)[~np.isnan(file_coords[f'longitude_{country_name}'].values)]
    lat_fin = (file_coords[f'latitude_{country_name}'].values)[~np.isnan(file_coords[f'latitude_{country_name}'].values)]
    
    # I want my reshaped grid starting from the upper left point of the country -> flip the latitude in descending order
    lat_fin = np.sort(lat_fin)[::-1]
    
    return lat_fin, lon_fin

In [4]:
lat_idw_new, lon_idw_new = files_reading(country)

### Loading the stations data:

In [5]:
def open_dataset(filepath):
    var_df = pd.read_csv(filepath)
    # Conversion needed for the monthly resampling
    var_df['time'] = pd.to_datetime(var_df['time'])
    return var_df

In [6]:
rad_df = open_dataset(data_path)

### Data Manipulation:

In [9]:
def data_manipulation(group):
    
    var_hourly_res   = group.resample('60min',on='time').mean()
    var_daily_res    = var_hourly_res.reset_index().resample('D',on='time').mean()
    
    #Conversion from W/m2 to kWh/m2/day
    var_daily_res['ssrd'] = (var_daily_res['ssrd']*24)/1000
    
    #monthly resampling
    var_monthly_res       = var_daily_res.reset_index().resample('M', on='time').mean()
    
    #monthly mean for every station (groups)
    var_months            = var_monthly_res.groupby(by=[var_monthly_res.index.month]).mean()
    
    #filtering to delete the outliers
    var_months_filtered   = var_months[var_months['ssrd'] > 0.5] 
    
    return var_months_filtered

In [10]:
rad_df_months = rad_df.groupby('name', group_keys=False).apply(lambda x: data_manipulation(x))

### IDW: execution

In [11]:
# Month selection
month = 1

In [12]:
#------------------------------------------------
# Applying IDW to IoT data to obtain a consistent map from punctual observations
#
# Parameters:
# - lat: type = array; latitude of locations of IoT sensors;
# - lon: type = array; longitude of locations of IoT sensors;
# - var: type = array; IoT data;
# - lat_new: type = array; latitude of final grid;
# - lon_new: type = array; longitude of final grid.
#
#------------------------------------------------
def apply_idw(pdf, lat_new, lon_new):

    lat = pdf['latitude'].values
    lon = pdf['longitude'].values
    var = pdf['ssrd'].values

    # Locations of the IoT observations
    iot_grid = np.array(list(zip(lat, lon)))

    # Grid for IoT observation projection
    grid_new = np.array([[x, y] for x in lat_new for y in lon_new])

    # Calculating IoT map with IDW
    iot_new = simple_idw(iot_grid, var, grid_new)

    # Reshaping results from array-like to matrix-like
    iot_reshape = iot_new.reshape(len(lat_new), len(lon_new))

    return iot_reshape

In [13]:
#--------------------------------------------
# Inverse Distance Weighting (IDW)
#
# Parameters:
# - pos: type = 2-dim array; position of locations of IoT sensors in the format (lat, lon);
# - var: type = array (length: pos); IoT data;
# - posi: type = 2-dim array; position of grid points in final grid the format (lat, lon).
#
#--------------------------------------------
def simple_idw(pos, z, posi):

    # Distance matrix
    dist = distance_matrix(pos, posi)**3

    # Weight matrix
    weights = 1.0 / dist

    # Change diagonal values to 1
    np.fill_diagonal(weights, 1)

    # Normalizing each column of the weight matrix
    weights = weights/np.nansum(weights, axis=0)

    # Calculating the results of the IDW method
    zi = np.dot(weights.T, z)

    return zi

In [14]:
#----------------------------------------------------------------------------------------------------
# This function considers a single month, applying the IDW functions defined at the beginning.
# Then it creates a dataframe with the reshaped variable, assigning the high-res final coordinates
# The file is saved
#----------------------------------------------------------------------------------------------------
def execute_idw_monthly(var_df_months, n_month, out_path, 
                        lat_idw_new, lon_idw_new,country):
    
    var_df_month = var_df_months[var_df_months.index == n_month]
    
    var_df_month_reshaped = apply_idw(var_df_month, lat_idw_new, lon_idw_new)
    
    var_idw_month_xr = xr.DataArray(var_df_month_reshaped, 
                                  coords=[("latitude", lat_idw_new),("longitude", lon_idw_new)]
                                   ).to_dataset(name='ssrd_reshaped').assign_attrs(units="kWh/m2/day")
    
    var_idw_month_xr.to_netcdf(out_path+f'ssrd_idw_{country}_m{n_month}.nc')
    
    
    return var_idw_month_xr

In [15]:
#Output: IDW of a single month - solar radiation (kWh/m2/day)
rad_idw_month = execute_idw_monthly(rad_df_months, month, output_path, 
                                          lat_idw_new, lon_idw_new,country)